<a href="https://colab.research.google.com/github/fboldt/aulasann/blob/main/aula05b%20-%20mlp%20real%20world%20dataset%20-%20keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [39]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelBinarizer
import numpy as np

def include_bias(X):
  return np.hstack((np.ones((X.shape[0], 1)), X))

def sign(a):
  return (a>=0)*2-1

class MultiLayer(BaseEstimator, ClassifierMixin):
  def __init__(self, max_iter=2000, learning_rate=0.001,
               n_hidden=[100]):
    self.max_iter = max_iter
    self.learning_rate = learning_rate
    self.n_hidden = n_hidden
    self.activ_func = np.tanh

  def forward(self, X):
    self.A = []
    self.Z = []
    AUX = X.copy()
    for W in self.Ws:
      self.A.append(include_bias(AUX))
      self.Z.append(self.A[-1] @ W)
      AUX = self.activ_func(self.Z[-1])
    return AUX

  def backward(self, y, logits):
    grads = []
    output_delta = logits - y
    grads.insert(0, self.A[-1].T @ output_delta)
    for i in range(len(self.Ws)-1, 0, -1):
      tanh_grad = 1 - np.tanh(self.Z[i-1])**2
      input_delta = output_delta @ self.Ws[i][1:].T * tanh_grad
      grads.insert(0, self.A[i-1].T @ input_delta)
      output_delta = input_delta.copy()
    for i in range(len(self.Ws)):
      self.Ws[i] -= grads[i] * self.learning_rate

  def fit(self, X, y):
    self.lb = LabelBinarizer()
    y = self.lb.fit_transform(y)
    if len(y.shape) == 1:
      y = y.reshape(-1, 1)
    self.Ws = []
    previous_output = X.shape[1]
    for n in self.n_hidden:
      self.Ws.append(np.random.uniform(-1, 1, (previous_output+1, n)))
      previous_output = n
    self.Ws.append(np.random.uniform(-1, 1, (previous_output+1, y.shape[1])))
    for _ in range(self.max_iter):
      logits = self.forward(X)
      self.backward(y, logits)
    return self

  def predict(self, X):
    logits = self.forward(X)
    ypred = sign(logits)
    return self.lb.inverse_transform(ypred)

model = MultiLayer()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.09166666666666666


In [40]:
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(max_iter=2000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9833333333333333


In [42]:
from sklearn.base import BaseEstimator, ClassifierMixin
from tensorflow import keras
import numpy as np

class SLPClassifier(BaseEstimator, ClassifierMixin):
  def __init__(self, max_iter=200):
    self.max_iter = max_iter
    self.model = None

  def fit(self, X, y):
    self.labels, ids = np.unique(y, return_inverse=True)
    input_shape = X.shape[1]
    output_shape = len(self.labels)
    y_hot = keras.utils.to_categorical(ids, num_classes=output_shape)
    self.model = keras.Sequential([
      keras.layers.Input(shape=(input_shape,)),
      keras.layers.Dense(output_shape, activation='softmax')
    ])
    self.model.compile(optimizer='adam',
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])
    self.model.fit(X, y_hot, epochs=self.max_iter, verbose=0)
    return self

  def predict_proba(self, X):
    return self.model.predict(X)

  def predict(self, X):
    y_pred = np.argmax(self.predict_proba(X), axis=1)
    return self.labels[y_pred]

model = SLPClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
0.9666666666666667


In [43]:
from sklearn.base import BaseEstimator, ClassifierMixin
from tensorflow import keras
import numpy as np

class SHLPClassifier(BaseEstimator, ClassifierMixin):
  def __init__(self, max_iter=200):
    self.max_iter = max_iter
    self.model = None

  def fit(self, X, y):
    self.labels, ids = np.unique(y, return_inverse=True)
    input_shape = X.shape[1]
    output_shape = len(self.labels)
    y_hot = keras.utils.to_categorical(ids, num_classes=output_shape)
    self.model = keras.Sequential([
      keras.layers.Input(shape=(input_shape,)),
      keras.layers.Dense(100, activation='relu'),
      keras.layers.Dense(output_shape, activation='softmax')
    ])
    self.model.compile(optimizer='adam',
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])
    self.model.fit(X, y_hot, epochs=self.max_iter, verbose=0)
    return self

  def predict_proba(self, X):
    return self.model.predict(X)

  def predict(self, X):
    y_pred = np.argmax(self.predict_proba(X), axis=1)
    return self.labels[y_pred]

model = SHLPClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
0.9805555555555555


In [45]:
from sklearn.base import BaseEstimator, ClassifierMixin
from tensorflow import keras
import numpy as np

class MLPClassifier(BaseEstimator, ClassifierMixin):
  def __init__(self, max_iter=200, n_hidden=[100]):
    self.max_iter = max_iter
    self.n_hidden = n_hidden
    self.model = None

  def fit(self, X, y):
    self.labels, ids = np.unique(y, return_inverse=True)
    input_shape = X.shape[1]
    output_shape = len(self.labels)
    y_hot = keras.utils.to_categorical(ids, num_classes=output_shape)

    self.model = keras.Sequential()
    self.model.add(keras.layers.Input(shape=(input_shape,)))
    for n in self.n_hidden:
      self.model.add(keras.layers.Dense(n, activation='relu'))
    self.model.add(keras.layers.Dense(output_shape, activation='softmax'))

    self.model.compile(optimizer='adam',
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])
    self.model.fit(X, y_hot, epochs=self.max_iter, verbose=0)
    return self

  def predict_proba(self, X):
    return self.model.predict(X)

  def predict(self, X):
    y_pred = np.argmax(self.predict_proba(X), axis=1)
    return self.labels[y_pred]

model = MLPClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
0.9861111111111112
